# **TD Python — Web Scraping "BOOKS" http://books.toscrape.com**

**Objectifs** <br>
+ envoyer une requête HTTP, <br>

+ analyser une page HTML avec BeautifulSoup, <br>

+ extraire des informations ciblées, <br>

+ nettoyer certaines données, <br>

+ parcourir plusieurs pages, <br>

+ sauvegarder les résultats dans un fichier CSV. <br>

## **1. Importer les bibliothèques**
On commence par charger les outils nécessaires : <br>

+ requests : pour télécharger la page web, <br>

+ BeautifulSoup : pour lire le HTML, <br>

+ urljoin : pour construire des URLs complètes, <br>

+ csv : pour sauvegarder les données, <br>

+ time : pour ajouter une pause entre les requêtes. <br>

In [93]:
# Completer le code

Un scraper fait souvent toujours les mêmes grandes choses : <br>

+ télécharger, <br>

+ lire le HTML, <br>

+ extraire, <br>

+ nettoyer, <br>

+ stocker. <br>

**Mini question** <br>

À quoi sert BeautifulSoup dans ce TP ? <br>

## **2. Définir les URLs de départ**

On définit : <br>

+ l’URL principale du site, <br>

+ l’URL de la première page à scraper. <br>

In [ ]:
BASE_URL = "http://books.toscrape.com/"
START_URL = urljoin(BASE_URL, "catalogue/page-1.html")

print("BASE_URL :", BASE_URL)
print("START_URL :", START_URL)

**Expliquer** <br>

+ urljoin --> <br>

## **3. Télécharger une page web**

Créer une fonction get_soup(url: str) qui : <br>

+ envoie une requête HTTP, <br>

+ vérifie que la réponse est correcte, <br>

+ transforme le HTML en objet BeautifulSoup. <br>

In [ ]:
# Code à compléter

In [ ]:
soup = get_soup(START_URL)
print(soup)
print(type(soup))

## **4. Explorer la structure HTML de la page**

Avant de scraper, il faut observer le HTML. <br>

In [ ]:
books = soup.select("article.product_pod")
print("Nombre de livres sur la page :", len(books))

**Puis**

In [ ]:
first_book = books[0]
first_book

In [ ]:
print(first_book.prettify()[:1000])

**Expliquer** <br>

+ select("article.product_pod") --> <br>

+ prettify() --> <br>

**Mini question** <br>

Combien de livres sont affichés sur une page ? <br>

## **5. Extraire les informations simples d’un livre**

Récupèrer les éléments utiles sur le premier livre : <br>

+ titre, <br>

+ prix, <br>

+ disponibilité, <br>

+ lien. <br>

In [99]:
# completer le code

**Expliquer** <br>

+ select_one(...) --> <br>

+ get_text(strip=True) --> <br>

**Mini activité** <br>

Faire repérer dans le HTML où se trouvent le titre et le prix. <br>

## **6. Nettoyer le prix**

Créer un fonction clean_price(price_text : str) -> float . Exemple on récupére comme texte, par exemple "£51.77" et puis en renvoie un réel 51.77 <br>
On veut un nombre float. <br>

In [ ]:
# completer le code

**test**

In [17]:
raw_price = price_tag.get_text(strip=True)
cleaned_price = clean_price(raw_price)

print("Prix brut :", raw_price)
print("Prix nettoyé :", cleaned_price)
print(type(cleaned_price))

Prix brut : Â£51.77
Prix nettoyé : 51.77
<class 'float'>


## **7. Convertir la note en entier**

La note n’est pas écrite comme un nombre. <br>
Elle apparaît dans une classe CSS du type : <br>

In [ ]:
<p class="star-rating Three">

On doit donc traduire "Three" en 3.

In [40]:
def clean_rating(article_tag) -> int:
    rating_map = {
        "One": 1,
        "Two": 2,
        "Three": 3,
        "Four": 4,
        "Five": 5,
    }

    rating_tag = article_tag.select_one("p.star-rating")
    if not rating_tag:
        return 0

    classes = rating_tag.get("class", [])
    for class_name in classes:
        if class_name in rating_map:
            return rating_map[class_name]
    return 0

In [ ]:
print("Note du premier livre :", clean_rating(first_book))

**Expliquer** <br>

+ get("class", []) --> <br>

**Mini question** <br>

Pourquoi la note n’est-elle pas récupérée avec get_text() ? <br>

## **8. Construire une fonction complète pour un livre**

Rassembler les extractions extract_book_data(book_tag)-> dict dans une seule fonction. <br>

In [101]:
# Completer le code

In [45]:
book_data = extract_book_data(first_book)
book_data

{'title': 'A Light in the Attic',
 'price': 51.77,
 'rating': 3,
 'availability': 'In stock',
 'product_url': 'http://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html'}

**Mini activité** <br>

Afficher le type de book_data et ses clés : <br>

In [ ]:
# completer le code

dict_keys(['title', 'price', 'rating', 'availability', 'product_url'])

## **9. Extraire tous les livres d’une page**

On boucle sur tous les livres présents sur une page. <br>

In [ ]:
# completer le code

Nombre de livres extraits : 20


[{'title': 'A Light in the Attic',
  'price': 51.77,
  'rating': 3,
  'availability': 'In stock',
  'product_url': 'http://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html'},
 {'title': 'Tipping the Velvet',
  'price': 53.74,
  'rating': 1,
  'availability': 'In stock',
  'product_url': 'http://books.toscrape.com/catalogue/tipping-the-velvet_999/index.html'},
 {'title': 'Soumission',
  'price': 50.1,
  'rating': 1,
  'availability': 'In stock',
  'product_url': 'http://books.toscrape.com/catalogue/soumission_998/index.html'}]

**Mini activité** <br>

Afficher seulement les titres : <br>

In [25]:
for book in books_page_1[:5]:
    print(book["title"])

A Light in the Attic
Tipping the Velvet
Soumission
Sharp Objects
Sapiens: A Brief History of Humankind


## **10. Trouver le lien vers la page suivante**

Le site a plusieurs pages. Il faut récupérer le bouton next. <br>

In [87]:
def get_next_page(soup: BeautifulSoup, current_url: str) -> str | None:
    next_tag = soup.select_one("li.next a")
    if not next_tag:
        return None
    return urljoin(current_url, next_tag["href"])

In [88]:
next_page = get_next_page(soup, START_URL)
print("Page suivante :", next_page)

Page suivante : http://books.toscrape.com/catalogue/catalogue/page-2.html


**Mini question** <br>

Pourquoi a-t-on besoin de current_url dans urljoin ? <br>

## **11. Scraper plusieurs pages**

On construit maintenant le vrai scraper complet. <br>

In [89]:
def scrape_books(start_url: str) -> list[dict]:
    books = []
    current_url = start_url

    while current_url:
        print(f"Scraping: {current_url}")
        soup = get_soup(current_url)

        book_tags = soup.select("article.product_pod")
        for book_tag in book_tags:
            books.append(extract_book_data(book_tag))

        current_url = get_next_page(soup, current_url)
        time.sleep(1)

    return books

In [90]:
all_books = scrape_books(START_URL)
print("Nombre total de livres :", len(all_books))
all_books[:5]

Scraping: http://books.toscrape.com/catalogue/page-1.html
Scraping: http://books.toscrape.com/catalogue/page-2.html
Scraping: http://books.toscrape.com/catalogue/page-3.html
Scraping: http://books.toscrape.com/catalogue/page-4.html
Scraping: http://books.toscrape.com/catalogue/page-5.html
Scraping: http://books.toscrape.com/catalogue/page-6.html
Scraping: http://books.toscrape.com/catalogue/page-7.html
Scraping: http://books.toscrape.com/catalogue/page-8.html
Scraping: http://books.toscrape.com/catalogue/page-9.html
Scraping: http://books.toscrape.com/catalogue/page-10.html
Scraping: http://books.toscrape.com/catalogue/page-11.html
Scraping: http://books.toscrape.com/catalogue/page-12.html
Scraping: http://books.toscrape.com/catalogue/page-13.html
Scraping: http://books.toscrape.com/catalogue/page-14.html
Scraping: http://books.toscrape.com/catalogue/page-15.html
Scraping: http://books.toscrape.com/catalogue/page-16.html
Scraping: http://books.toscrape.com/catalogue/page-17.html
Scrapi

[{'title': 'A Light in the Attic',
  'price': 51.77,
  'rating': 3,
  'availability': 'In stock',
  'product_url': 'http://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html'},
 {'title': 'Tipping the Velvet',
  'price': 53.74,
  'rating': 1,
  'availability': 'In stock',
  'product_url': 'http://books.toscrape.com/catalogue/tipping-the-velvet_999/index.html'},
 {'title': 'Soumission',
  'price': 50.1,
  'rating': 1,
  'availability': 'In stock',
  'product_url': 'http://books.toscrape.com/catalogue/soumission_998/index.html'},
 {'title': 'Sharp Objects',
  'price': 47.82,
  'rating': 4,
  'availability': 'In stock',
  'product_url': 'http://books.toscrape.com/catalogue/sharp-objects_997/index.html'},
 {'title': 'Sapiens: A Brief History of Humankind',
  'price': 54.23,
  'rating': 5,
  'availability': 'In stock',
  'product_url': 'http://books.toscrape.com/catalogue/sapiens-a-brief-history-of-humankind_996/index.html'}]

**Expliquer** <br>

while current_url --> <br>

time.sleep(1) -->  <br>

**Mini activité** <br>

Compter le nombre total de livres, <br>

Afficher le dernier livre extrait. <br>

## **12. Sauvegarder dans un CSV**

Une fois les données extraites, on les stocke dans un fichier CSV. <br>

In [91]:
def save_to_csv(data: list[dict], filename: str = "books.csv") -> None:
    if not data:
        print("Aucune donnée à enregistrer.")
        return

    fieldnames = data[0].keys()
    with open(filename, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(data)

    print(f"Fichier enregistré : {filename}")

In [92]:
save_to_csv(all_books, "books.csv")

Fichier enregistré : books.csv


**Expliquer** <br>

DictWriter -->  <br>

writeheader() --> <br>

writerows() --> <br>

**Mini activité** <br>

Faire ouvrir le fichier CSV et vérifier les colonnes. <br>

**Exercice 1** <br>

Modifier le code pour extraire seulement : <br>

+ le titre, <br>

+ le prix, <br>

+ la note. <br>

**Exercice 2** <br>

Ajouter une colonne in_stock avec True ou False. <br>

**Exercice 3** <br>

Afficher uniquement les livres dont la note est supérieure ou égale à 4.



**Exercice 4** <br>

Compter le nombre de livres par note.



**Exercice 5** <br>

Sauvegarder uniquement les livres coûtant plus de 30 livres.